In [6]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, random_split
import os

In [7]:
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 5
NUM_CLASSES = 43
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [8]:
stats = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

In [10]:
full_dataset = datasets.GTSRB(root="data", split="train", download=True, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_set, val_set = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

In [11]:
def create_model(task_type):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

    if task_type == "task1_frozen":
        # Freeze all backbone layers
        for param in model.features.parameters():
            param.requires_grad = False

    elif task_type == "task2_fine_tune_all":
        # All layers trainable
        for param in model.parameters():
            param.requires_grad = True

    elif task_type == "task3_partial_freeze":
        # Freeze up to a specific block
        for i, child in enumerate(model.features.children()):
            if i < 4:
                for param in child.parameters():
                    param.requires_grad = False
            else:
                for param in child.parameters():
                    param.requires_grad = True

    # Replace classification head
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
    return model.to(DEVICE)

In [12]:
def train_engine(task_name):
    print(f"\n--- Starting {task_name} ---")
    model = create_model(task_name)

    # Optimizer (Adam) - only passing parameters that are not frozen
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(EPOCHS):

        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total_train += labels.size(0)
            correct_train += predicted.eq(labels).sum().item()

        # --- Validation Phase ---
        model.eval()
        running_val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)

                running_val_loss += loss.item()
                _, predicted = outputs.max(1)
                total_val += labels.size(0)
                correct_val += predicted.eq(labels).sum().item()


        train_acc = 100. * correct_train / total_train
        val_acc = 100. * correct_val / total_val
        avg_train_loss = running_loss / len(train_loader)
        avg_val_loss = running_val_loss / len(val_loader)

        print(f"Epoch [{epoch+1}/{EPOCHS}] "
              f"| Train Loss: {avg_train_loss:.4f}, Acc: {train_acc:.2f}% "
              f"| Val Loss: {avg_val_loss:.4f}, Acc: {val_acc:.2f}%")


    torch.save(model.state_dict(), f"efficientnet_{task_name}.pth")
    print(f"Task {task_name} finished and model saved.")

In [13]:
if __name__ == "__main__":
    all_tasks = ["task1_frozen", "task2_fine_tune_all", "task3_partial_freeze"]

    for current_task in all_tasks:
        train_engine(current_task)


--- Starting task1_frozen ---
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 92.0MB/s]


Epoch [1/5] | Train Loss: 1.4462, Acc: 64.19% | Val Loss: 0.7386, Acc: 81.53%
Epoch [2/5] | Train Loss: 0.7475, Acc: 80.42% | Val Loss: 0.5335, Acc: 87.05%
Epoch [3/5] | Train Loss: 0.5793, Acc: 83.96% | Val Loss: 0.5324, Acc: 89.17%
Epoch [4/5] | Train Loss: 0.5045, Acc: 85.76% | Val Loss: 0.3707, Acc: 90.80%
Epoch [5/5] | Train Loss: 0.4554, Acc: 86.90% | Val Loss: 0.3262, Acc: 91.65%
Task task1_frozen finished and model saved.

--- Starting task2_fine_tune_all ---
Epoch [1/5] | Train Loss: 0.2538, Acc: 93.22% | Val Loss: 0.0319, Acc: 99.21%
Epoch [2/5] | Train Loss: 0.0470, Acc: 98.74% | Val Loss: 0.0082, Acc: 99.76%
Epoch [3/5] | Train Loss: 0.0246, Acc: 99.29% | Val Loss: 0.0084, Acc: 99.74%
Epoch [4/5] | Train Loss: 0.0245, Acc: 99.31% | Val Loss: 0.0127, Acc: 99.61%
Epoch [5/5] | Train Loss: 0.0206, Acc: 99.52% | Val Loss: 0.0043, Acc: 99.92%
Task task2_fine_tune_all finished and model saved.

--- Starting task3_partial_freeze ---
Epoch [1/5] | Train Loss: 0.2565, Acc: 93.24% | 